# 🎛️ Analog Resurrection

## Project 4 · Open-Ended · Direction 2 (Technical)

Every fact in this introduction is real. Go ahead and check.

[Alexander "Howard" Dumble](https://en.wikipedia.org/wiki/Alexander_Dumble) hand-built somewhere
around three hundred guitar amplifiers in his lifetime, each one individually voiced for its
player — Stevie Ray Vaughan, Robben Ford, Larry Carlton, Carlos Santana. John Mayer has toured with
a Dumble Steel String Singer. And because Dumble did not want to be copied, he **potted his circuit
boards in opaque epoxy** — "goop," collectors call it — so that even if you owned one, you couldn't
see how it worked. He died in January 2022, having never published his designs. Surviving amps now
sell for six figures, and a small community of engineers carefully dissolves the goop on broken
units, tracing the circuits, trying to recover the sound before the originals age out of existence.

You don't need boutique mythology to find this story, though. The most recorded equalizer in
history is probably the [**Neve 1073**](https://en.wikipedia.org/wiki/Neve_1073), a console channel
module designed in 1970 under [Rupert Neve](https://en.wikipedia.org/wiki/Rupert_Neve) (1926–2021).
Fifty-five years later, studios still pay thousands of dollars per module for the originals. Most
of us will never touch one.

And yet: you can buy a 1073 — or a Pultec, or an SSL bus compressor, or a Fairchild 670 worth more
than a house — as a **plugin**, for a hundredth of the price. How? Companies like Universal Audio
and Waves do professionally what the Dumble de-goopers do out of love: **analog resurrection**.
Take the circuit. Derive its continuous-time transfer function $H(s)$. Then carry it across the
bridge into the digital world, where it becomes a **difference equation** your laptop can run. The
workhorse of that crossing is the **bilinear transform** — the exact technique in this notebook,
and the same one used in the classic academic treatment of the problem (David Yeh & Julius Smith,
*"Discretization of the '59 Fender Bassman Tone Stack,"* DAFx 2006 — a real paper about digitally
resurrecting a real amplifier's EQ circuit).

Today, you join the resurrection business.

### Your mission

You will build an **analog-to-digital EQ compiler** — front end: analog filter sections; back end:
a running digital filter — and use it to resurrect a channel EQ built in the image of the Neve
1073. Four build tasks, each with a checkpoint and step-by-step guidance:

1. **Stock the parts bin** — implement the standard analog prototype sections (all formulas
   provided; a peaking section is done for you as a worked example).
2. **Build the bridge** — write `bilinear_transform`, for sections of **any order**, with
   **pre-warping** so every band lands at exactly the right frequency.
3. **Reassemble the channel strip** — write `combine_cascade`, fusing the sections into one
   difference equation.
4. **Make it run** — write `apply_filter`, your own difference-equation engine.

Then the payoff, mostly provided: **prove it** (a four-way verification), **break it** (a
numerical experiment where a filter explodes), and **hear it** (a listening test).

> ⚠️ **This notebook is a launchpad, not the whole assignment.** The full requirements,
> deliverables (`TECHNICAL.md`, demo video), and submission instructions live on the Project 4 page
> of the course site.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq

SAMPLE_RATE = 44100  # the resurrected EQ will run at CD quality
FREQS = np.geomspace(20, 20000, 2048)  # log-spaced frequency grid (Hz) for all plots


## 🔩 The unit on the bench

The real 1073 gives you three EQ bands plus a high-pass filter, with switched frequency choices.
Our resurrection target uses its exact architecture and knob values:

| Section | Type | Our setting | Drawn from the real 1073's options |
|---|---|---|---|
| *Rumble filter* | high-pass, **3rd-order Butterworth** (18 dB/oct) | 50 Hz | 50 / 80 / 160 / 300 Hz |
| *Low shelf* | shelving | 110 Hz, +4 dB | 35 / 60 / 110 / 220 Hz |
| *Presence* | peaking | 3.2 kHz, +3 dB, Q 1.0 | 0.36 / 0.7 / 1.6 / 3.2 / 4.8 / 7.2 kHz |
| *Air* | shelving | 12 kHz, +4 dB | fixed at 12 kHz |

(To be precise: we're borrowing the 1073's *filter architecture*, not claiming a component-level
model of its transformers and Class-A gain stages. That rabbit hole is deeper — and it's half the
mystique.)

Note the rumble filter. The 1073's high-pass rolls off at **18 dB per octave**, which makes it a
**third-order** section. Keep an eye on it: it's the reason several shortcuts in this assignment
don't work, and your compiler will have to earn its keep.


In [ ]:
# Our resurrection target: a channel EQ in the image of the Neve 1073.
# This list is the input format your compiler consumes — and your compiler must
# work for ANY such list, not just this one.
CHANNEL_EQ = [
    dict(name="Rumble filter", kind="highpass",   f0=50.0,    order=3),
    dict(name="Low shelf",     kind="low_shelf",  f0=110.0,   gain_db=+4.0),
    dict(name="Presence",      kind="peak",       f0=3200.0,  gain_db=+3.0, q=1.0),
    dict(name="Air",           kind="high_shelf", f0=12000.0, gain_db=+4.0),
]

for band in CHANNEL_EQ:
    print(f"{band['name']:>14}: {band}")


## 🧰 Task 1 — Stock the parts bin: `analog_prototype`

Analog filter design has a standard catalog of low-order building blocks, and the industry's
reference card for them is Robert Bristow-Johnson's
[**Audio EQ Cookbook**](https://www.w3.org/TR/audio-eq-cookbook/) — a real document, published as a
W3C note, cited in DSP code everywhere. It gives each section as a **normalized analog prototype**:
a transfer function whose *critical frequency* — the center of a peak, the midpoint of a shelf,
the cutoff of a high-pass — sits at exactly $\omega = 1$ rad/s. Normalizing keeps the numbers
tame, and (as you'll see in Task 2) makes the trip to the digital world remarkably clean. To
reconstruct a prototype's *real* analog response at critical frequency $f_0$, evaluate it at
$s = j\,(f / f_0)$.

We represent a prototype as two coefficient arrays `B` and `A`, **highest power of $s$ first** —
length 3 for a second-order section, length 4 for the third-order rumble filter.

**Your job:** transcribe the catalog into code. All the formulas you need are right here, with
$A_g = 10^{\text{gain\_db}/40}$ (shelves and peaks split their dB gain this way):

- **Peaking** *(implemented for you as a worked example)*:
  $\;H(s) = \dfrac{s^2 + (A_g/Q)\,s + 1}{s^2 + \big(1/(A_g Q)\big)\,s + 1}$
- **Low shelf**:
  $\;H(s) = \dfrac{A_g\big(s^2 + (\sqrt{A_g}/Q)\,s + A_g\big)}{A_g\,s^2 + (\sqrt{A_g}/Q)\,s + 1}$
- **High shelf**:
  $\;H(s) = \dfrac{A_g\big(A_g\,s^2 + (\sqrt{A_g}/Q)\,s + 1\big)}{s^2 + (\sqrt{A_g}/Q)\,s + A_g}$
- **Butterworth high-pass of order $n$**:
  $\;H(s) = \dfrac{s^n}{B_n(s)}$, where the Butterworth polynomials are
  $B_1 = s + 1$, $\;B_2 = s^2 + \sqrt{2}\,s + 1$, $\;B_3 = s^3 + 2s^2 + 2s + 1$.
  (Support orders 1–3; note the arrays have length $n{+}1$, so your prototypes are no longer all
  second-order — which is the point.)

Careful, methodical transcription is the whole task: get the coefficient *order* right
(highest power of $s$ first) and distribute the $A_g$ out front into the numerator array.

> ⚠️ **A warning about the cookbook:** it also tabulates final *digital* biquad coefficients.
> Copying those defeats the assignment twice over — your pipeline must go through the analog domain
> and your own bilinear transform (that's what's graded, and what the oral-quiz policy probes), and
> the cookbook's biquad tables *cannot* produce the third-order rumble filter anyway.


In [ ]:
def analog_prototype(kind, gain_db=0.0, q=0.707, order=1):
    """One normalized analog section from the standard catalog.

    Returns (B, A): coefficient arrays of H(s), highest power of s first,
    normalized so the section's critical frequency is at omega = 1 rad/s.
    Array length is (section order + 1) — e.g. length 3 for second-order,
    length 4 for a third-order Butterworth high-pass.
    """
    Ag = 10 ** (gain_db / 40)
    if kind == "peak":  # worked example: H(s) = (s^2 + (Ag/q)s + 1) / (s^2 + (1/(Ag q))s + 1)
        return np.array([1.0, Ag / q, 1.0]), np.array([1.0, 1.0 / (Ag * q), 1.0])
    elif kind == "low_shelf":
        # YOUR CODE HERE — numerator Ag*(s^2 + (sqrt(Ag)/q)s + Ag),
        #                  denominator Ag*s^2 + (sqrt(Ag)/q)s + 1
        raise NotImplementedError
    elif kind == "high_shelf":
        # YOUR CODE HERE — numerator Ag*(Ag*s^2 + (sqrt(Ag)/q)s + 1),
        #                  denominator s^2 + (sqrt(Ag)/q)s + Ag
        raise NotImplementedError
    elif kind == "highpass":
        # YOUR CODE HERE — H(s) = s^order / B_order(s) for orders 1..3.
        # Numerator: [1, 0, ..., 0] (length order+1). Denominator: the Butterworth
        # polynomial coefficients from the list in the cell above.
        raise NotImplementedError
    raise ValueError(f"unknown section kind: {kind!r}")


### The measurement bench (provided)

Helpers that evaluate a transfer function's frequency response by plugging in points along the
frequency axis — $s = j\,(f/f_0)$ for a normalized analog prototype, $z = e^{\,j 2\pi f / f_s}$
for a digital filter — plus a helper that runs your `analog_prototype` over a whole EQ
configuration.


In [ ]:
def analog_response(B, A, f0, freqs):
    """Frequency response of a normalized analog section with critical frequency f0 (Hz),
    evaluated at freqs (Hz). Returns a complex array."""
    s = 1j * (np.asarray(freqs) / f0)
    return np.polyval(B, s) / np.polyval(A, s)


def digital_response(b, a, freqs, sample_rate):
    """Frequency response of a digital filter H(z) = (sum_k b[k] z^-k) / (sum_k a[k] z^-k),
    evaluated at freqs (Hz). Works for any filter order. Returns a complex array."""
    zinv = np.exp(-2j * np.pi * np.asarray(freqs) / sample_rate)
    return np.polyval(np.asarray(b)[::-1], zinv) / np.polyval(np.asarray(a)[::-1], zinv)


def db(H):
    """Magnitude in decibels."""
    return 20 * np.log10(np.maximum(np.abs(H), 1e-12))


def eq_sections(config):
    """Compile an EQ config into a list of (name, B, A, f0) analog sections."""
    sections = []
    for band in config:
        params = {k: v for k, v in band.items() if k not in ("name", "f0")}
        B, A = analog_prototype(**params)
        sections.append((band["name"], B, A, band["f0"]))
    return sections


### Checkpoint: does the parts bin check out?

Facts your prototypes must reproduce (all at the critical frequency, $f = f_0$):

- A **peaking** section reads its full gain at $f_0$.
- A **shelf** reads exactly *half* its dB gain at $f_0$ — a shelf's critical frequency is its
  midpoint, where it has climbed halfway. (The full gain arrives in the flat region beyond.)
- A **Butterworth high-pass** of *any* order reads exactly $-3.01$ dB at its cutoff. That's the
  Butterworth signature.

If the asserts pass, you get your first look at the target: the analog response of the whole
channel strip. Burn this curve into your memory — by the end of the notebook, your digital filter
has to trace it.


In [ ]:
sections = eq_sections(CHANNEL_EQ)

expected_at_f0 = {"peak": lambda band: band.get("gain_db", 0.0),
                  "low_shelf": lambda band: band.get("gain_db", 0.0) / 2,
                  "high_shelf": lambda band: band.get("gain_db", 0.0) / 2,
                  "highpass": lambda band: -20 * np.log10(np.sqrt(2))}
for band, (name, B, A, f0) in zip(CHANNEL_EQ, sections):
    got = db(analog_response(B, A, f0, np.array([f0])))[0]
    want = expected_at_f0[band["kind"]](band)
    assert abs(got - want) < 0.01, (
        f"{name}: expected {want:+.2f} dB at f0={f0:.0f} Hz, got {got:+.2f} dB — "
        f"check your {band['kind']!r} prototype")
    print(f"  ✓ {name:>14}: {got:+.2f} dB at {f0:.0f} Hz, as the catalog demands")

H_truth = np.ones_like(FREQS, dtype=complex)
for name, B, A, f0 in sections:
    H_truth *= analog_response(B, A, f0, FREQS)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogx(FREQS, db(H_truth), color="darkgoldenrod", lw=2.5)
for band in CHANNEL_EQ:
    ax.axvline(band["f0"], color="gray", ls=":", lw=0.8)
    ax.annotate(band["name"], (band["f0"], ax.get_ylim()[0]), rotation=90,
                fontsize=8, color="gray", va="bottom", ha="right")
ax.set_title("The target: analog response of the 1073-style channel EQ")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("gain (dB)")
ax.grid(alpha=0.3, which="both")
plt.show()


## 🌉 Task 2 — Build the bridge: `bilinear_transform`

The bridge between the two worlds is the **bilinear transform**. It converts a continuous-time
transfer function $H(s)$ into a discrete-time $H(z)$ by substituting

$$s \;=\; \frac{2}{T}\cdot\frac{1 - z^{-1}}{1 + z^{-1}}, \qquad T = \frac{1}{f_s}.$$

The bridge is *sturdy* — it always maps a stable analog filter to a stable digital filter — but it
is not *straight*. The analog frequency axis is infinite; the digital one ends at Nyquist
($f_s/2$). The bilinear transform squeezes the entire infinite analog axis into that finite range,
compressing frequencies more and more as they approach Nyquist. This is **frequency warping**.

March a section across the bridge naively and it arrives at the wrong address: at 44.1 kHz, the
*Air* shelf aimed at 12 kHz would land at about **9.9 kHz** — more than two kilohertz flat. On the
most famous treble band in recording history. Unacceptable.

### Pre-warping: aim high, land on target

The fix is to *aim high*: before crossing, move the analog critical frequency to the **pre-warped**
frequency

$$\Omega \;=\; \frac{2}{T}\tan\!\left(\frac{\omega_0 T}{2}\right), \qquad \omega_0 = 2\pi f_0,$$

chosen so the warp of the bridge lands it *exactly* at $f_0$ on the digital side.

Here is where the catalog's normalization pays off. A prototype's critical frequency sits at
$s = j \cdot 1$, and placing it at analog frequency $\Omega$ just means substituting
$s \to s/\Omega$. Chain that with the bilinear substitution and the whole crossing collapses into
**one clean substitution** into the prototype:

$$s \;\leftarrow\; K \cdot \frac{1 - z^{-1}}{1 + z^{-1}},
\qquad K = \frac{2/T}{\Omega} = \frac{1}{\tan(\pi f_0 / f_s)}.$$

### Your job — and why it must handle any order

For a prototype of order $N$ (coefficient arrays of length $N{+}1$), substitute and clear
fractions by multiplying numerator and denominator by $(1+z^{-1})^N$. Each term of the polynomial
then contributes

$$\text{coeff}_n \cdot K^n \,(1 - z^{-1})^n\,(1 + z^{-1})^{N-n}$$

for its power $s^n$, and the results sum into numerator and denominator polynomials in $z^{-1}$ of
length $N{+}1$. Finally, normalize both so that $a_0 = 1$.

For $N = 2$ you could expand this by hand — the classic biquad derivation, three lines of algebra.
But our rumble filter is **third-order**, and hand-expansion stops being fun right about there.
Notice instead that every factor above is a small polynomial, and multiplying polynomials is
`np.convolve`. **Write the loop** — the recipe is spelled out step by step in the cell below, and
we've provided a `_poly_power` helper so the polynomial bookkeeping is one call away. A correct
general implementation is under fifteen lines, and it will swallow any section your compiler ever
meets.


In [ ]:
def _poly_power(p, k):
    """(provided helper) The polynomial p raised to the k-th power, via repeated np.convolve.
    Polynomials are coefficient arrays in z^-1, constant term first; p**0 is [1.0]."""
    out = np.array([1.0])
    for _ in range(k):
        out = np.convolve(out, p)
    return out


def bilinear_transform(B, A, f0, sample_rate):
    """Carry one analog section across the bridge to the digital world.

    Args:
        B, A: coefficient arrays of a *normalized* analog prototype H(s)
            (highest power of s first, critical frequency at omega = 1),
            of any order — length 3 for a biquad, length 4 for third-order, etc.
        f0: the section's true critical frequency in Hz. Pre-warp this!
        sample_rate: the digital sample rate in Hz.

    Returns:
        (b, a): numpy arrays of digital filter coefficients in powers of z^-1
            (constant term first), same length as the section's order + 1,
            normalized so that a[0] == 1.
    """
    # Recipe:
    #   1. N = the section's order. If B and A have different lengths, pad the
    #      shorter one with zeros AT THE FRONT (the front holds the highest powers
    #      of s) so both have length N + 1.
    #   2. K = 1 / tan(pi * f0 / sample_rate)   — the pre-warped substitution constant.
    #   3. Start b and a as zero arrays of length N + 1. Coefficient index i of the
    #      prototype carries the power n = N - i of s, and contributes the polynomial
    #          coeff * K**n * (1 - z^-1)**n * (1 + z^-1)**(N - n).
    #      Build (1 - z^-1)**n with _poly_power([1.0, -1.0], n), likewise for
    #      (1 + z^-1)**(N - n) with [1.0, 1.0], multiply the two with np.convolve
    #      (the product has length N + 1), scale, and add into the running b or a.
    #   4. Normalize: divide BOTH b and a by a[0].
    # YOUR CODE HERE
    raise NotImplementedError("The bridge isn't going to build itself.")


### Checkpoint: four sections across the bridge

If your transform is right, each section's digital curve should sit on top of its analog
counterpart — and at $f_0$ they must agree *exactly*. That is the pre-warp promise: perfect
alignment at the critical frequency.

You will also see the digital curves peel away from the analog ones as they approach Nyquist —
most visibly for the *Air* shelf. That's frequency warping squeezing infinity into 22.05 kHz. Not
a bug; worth a sentence in your `TECHNICAL.md`.


In [ ]:
digital_sections = []
fig, ax = plt.subplots(figsize=(10, 3.5))
for i, (name, B, A, f0) in enumerate(sections):
    b, a = bilinear_transform(B, A, f0, SAMPLE_RATE)
    assert len(b) == max(len(B), len(A)), (
        f"{name}: expected {max(len(B), len(A))} digital coefficients, got {len(b)} — "
        "your transform must preserve the section's order")
    digital_sections.append((b, a))

    color = plt.get_cmap("tab10")(i)
    ax.semilogx(FREQS, db(analog_response(B, A, f0, FREQS)), color=color, ls="--", lw=1.2)
    ax.semilogx(FREQS, db(digital_response(b, a, FREQS, SAMPLE_RATE)),
                color=color, lw=1.8, label=name)

    gain_analog = db(analog_response(B, A, f0, np.array([f0])))[0]
    gain_digital = db(digital_response(b, a, np.array([f0]), SAMPLE_RATE))[0]
    assert abs(gain_analog - gain_digital) < 0.01, (
        f"{name}: digital gain at f0 is {gain_digital:.3f} dB but the analog section "
        f"says {gain_analog:.3f} dB — check your pre-warping!")
    print(f"  ✓ {name:>14} across the bridge — {gain_digital:+.2f} dB at {f0:.0f} Hz, on target")

ax.set_title("Section by section: analog original (dashed) vs. digital resurrection (solid)")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("gain (dB)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=8)
plt.show()
print("All four sections made it across. 🌉")


## 🧩 Task 3 — Reassemble the channel strip: `combine_cascade`

Inside the module, the sections are wired **in series**: rumble filter into low shelf into
presence into air, and out. A series chain is a **product** of transfer functions:

$$H_{\text{EQ}}(z) = H_1(z)\cdot H_2(z)\cdot H_3(z)\cdot H_4(z).$$

Each $H_i(z)$ is a ratio of polynomials in $z^{-1}$, so the product's numerator is the product of
all the numerators, and likewise for denominators. And multiplying polynomials is an operation you
already know by another name: **convolution of their coefficient arrays** — `np.convolve`, the same
function from the autograded section, moonlighting as algebra.

Implement `combine_cascade`: fuse the list of digital sections into the single pair `(b, a)` of one
big difference equation. (Ours comes out **9th order**: $3 + 2 + 2 + 2$. This one is short — a few
lines.)


In [ ]:
def combine_cascade(digital_sections):
    """Fuse a cascade of digital filter sections into one difference equation.

    Args:
        digital_sections: a list of (b, a) coefficient-array pairs, one per section
            (any orders — they need not all match).

    Returns:
        (b, a): the numerator and denominator coefficient arrays of the single
            combined H(z), i.e. the full digital EQ as one difference equation.
    """
    # Recipe: start from b = [1.0] and a = [1.0], then np.convolve each section's
    # numerator into b and each section's denominator into a. (Numerators multiply
    # with numerators, denominators with denominators — don't mix them.)
    # YOUR CODE HERE
    raise NotImplementedError("Four sections, one channel strip.")


## ⚙️ Task 4 — Make it run: `apply_filter`

A transfer function $H(z)$ is a promise. A **difference equation** is a machine. With $a_0$
normalized to 1, the machine is:

$$y[n] = b_0 x[n] + b_1 x[n-1] + \dots + b_M x[n-M] \;-\; a_1 y[n-1] - \dots - a_M y[n-M].$$

Write `apply_filter`, your own engine that runs this equation over a signal, one sample at a time,
for **any** filter order. No `scipy.signal`, no library filters — the whole point of the compiler
is that its output is a handful of numbers plus *this* loop. Implement the equation exactly as
written above (the recipe in the cell spells it out); any sample with a negative index is simply
zero — the filter starts at rest.

(Yes, a Python loop over 44,100 samples per second is slow — the listening test later will take a
few seconds per filter. Production EQs run the identical arithmetic in optimized C. Same math,
different horsepower — feel free to say so in your video.)

### Then prove the machine matches the math

A running filter can be **measured** like a physical device: feed it a unit impulse
($x = [1, 0, 0, \dots]$), collect the output — the *impulse response* — and take its FFT. By the
convolution theorem you met in the autograded section, that spectrum *is* the filter's frequency
response. The checkpoint below measures your engine this way and lays the measurement over the
theoretical $H(z)$ curve. Math, meet machine.


In [ ]:
def apply_filter(b, a, x):
    """Run the difference equation defined by (b, a) over signal x.

    Args:
        b, a: digital filter coefficient arrays (any order; a[0] need not be 1 —
            normalize inside).
        x: the input signal, a 1-D array of samples.

    Returns:
        y: the filtered signal, same length as x (a numpy array).
    """
    # Recipe (direct form I — the difference equation verbatim):
    #   1. Normalize: divide every entry of b and of a by a[0].
    #   2. Make an output array y of zeros, same length as x.
    #   3. For each n in range(len(x)):
    #        y[n] = sum over k of  b[k] * x[n - k]   (for the k where n - k >= 0)
    #             - sum over k >= 1 of  a[k] * y[n - k]   (for the k where n - k >= 0)
    #      The y values you need on the right-hand side are ones you've already
    #      computed — that's what makes the filter recursive.
    #   Tip: plain Python floats in the inner loop are fine (and faster than tiny
    #   numpy operations). Correctness first; the checkpoint will catch any slip.
    # YOUR CODE HERE
    raise NotImplementedError("A transfer function is a promise. Keep it.")


In [ ]:
# Measure the running machine: impulse in, FFT of what comes out.
b_test, a_test = digital_sections[2]  # the Presence peak
impulse = np.zeros(8192)
impulse[0] = 1.0
ir = apply_filter(b_test, a_test, impulse)

fft_freqs = np.fft.rfftfreq(len(ir), 1 / SAMPLE_RATE)
H_measured = np.fft.rfft(ir)
H_theory = digital_response(b_test, a_test, fft_freqs, SAMPLE_RATE)

band = (fft_freqs >= 20) & (fft_freqs <= 20000)
dev = np.max(np.abs(db(H_measured[band]) - db(H_theory[band])))

fig, ax = plt.subplots(figsize=(10, 3))
ax.semilogx(fft_freqs[band], db(H_theory[band]), color="darkgoldenrod", lw=3, alpha=0.5,
            label="theory: $H(z)$ on the unit circle")
ax.semilogx(fft_freqs[band], db(H_measured[band]), color="tab:blue", lw=1.2, ls="--",
            label="measured: FFT of your engine's impulse response")
ax.set_title("Math, meet machine (Presence section)")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("gain (dB)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=9)
plt.show()

print(f"max |measured - theory| deviation: {dev:.2e} dB")
assert dev < 0.01, "Your engine's measured response disagrees with H(z) — check apply_filter."
print("The machine keeps the promise. ⚙️")


## 📈 Task 5 — The proof

Four curves go on one plot (this cell is provided — it runs on your functions):

- **(a) The analog target** — the cascade of analog sections from Task 1.
- **(b) Your digital cascade** — the product of your digital sections' responses.
- **(c) Your combined difference equation** — the single 9th-order $H(z)$ from `combine_cascade`.
- **(d) The measurement** — impulse through your `apply_filter` running the combined filter, FFT'd.

Curves **(b)**, **(c)**, and **(d)** are mathematically the same filter, so they should agree to
within a few hundredths of a dB. Tellingly, though, (b) and (c) will *not* match to full machine
precision: merely *evaluating* one big 9th-order polynomial leaks real floating-point accuracy
compared to multiplying four small ones, worst at low frequencies where the rumble filter's roots
crowd together near $z = 1$. Tuck that observation away — it's your first clue for Task 6. All
three should hug curve **(a)** across the audible range, drifting away only near Nyquist, where
warping compresses the frequency axis.

If all four line up: the resurrection is verified, by theory and by measurement. These are the
plots your `TECHNICAL.md` needs.


In [ ]:
b_full, a_full = combine_cascade(digital_sections)
print(f"combined difference equation: order {len(a_full) - 1}")

# (b) product of the digital sections' responses
H_cascade = np.ones_like(FREQS, dtype=complex)
for b, a in digital_sections:
    H_cascade *= digital_response(b, a, FREQS, SAMPLE_RATE)

# (c) the single combined filter, evaluated
H_combined = digital_response(b_full, a_full, FREQS, SAMPLE_RATE)

# (d) the single combined filter, MEASURED running in your engine
impulse = np.zeros(16384)
impulse[0] = 1.0
ir_full = apply_filter(b_full, a_full, impulse)
fft_freqs = np.fft.rfftfreq(len(ir_full), 1 / SAMPLE_RATE)
H_measured = np.fft.rfft(ir_full)
band = (fft_freqs >= 20) & (fft_freqs <= 20000)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.semilogx(FREQS, db(H_truth), color="darkgoldenrod", lw=3.5, alpha=0.5,
            label="(a) analog target")
ax.semilogx(FREQS, db(H_cascade), color="tab:blue", lw=1.6, ls="--",
            label="(b) digital cascade")
ax.semilogx(FREQS, db(H_combined), color="tab:red", lw=1.2, ls=":",
            label="(c) combined difference equation")
ax.semilogx(fft_freqs[band], db(H_measured[band]), color="tab:green", lw=0.9, alpha=0.8,
            label="(d) measured impulse response")
ax.set_title("The resurrection, verified four ways")
ax.set_xlabel("frequency (Hz)")
ax.set_ylabel("gain (dB)")
ax.grid(alpha=0.3, which="both")
ax.legend(fontsize=9)
plt.show()

dev_bc = np.max(np.abs(db(H_cascade) - db(H_combined)))
dev_measured = np.max(np.abs(db(H_measured[band])
                             - db(digital_response(b_full, a_full, fft_freqs[band], SAMPLE_RATE))))
dev_analog = np.max(np.abs(db(H_truth) - db(H_cascade))[FREQS < 5000])
print(f"max |cascade - combined| deviation: {dev_bc:.2e} dB")
print("  (small, but noticeably not zero — the 9th-order polynomial is already leaking precision)")
print(f"max |measured - theory| deviation: {dev_measured:.2e} dB")
print(f"max |analog - digital| deviation below 5 kHz: {dev_analog:.3f} dB")
assert dev_bc < 0.1, "Combined filter disagrees with the cascade — check combine_cascade."
assert dev_measured < 1e-2, "Measurement disagrees with theory — check apply_filter."
print("Verified. The channel strip lives again. 🎛️⚡")


## 💥 Task 6 — Break it (the fragility experiment)

You just proved the cascade and the combined 9th-order filter are the same filter. So why does
every EQ that has ever shipped — hardware or plugin — *run* as a cascade of sections **no higher
than second order**?

Real filters don't get to keep their coefficients to fifteen decimal places. Early digital mixing
consoles ran on 16- and 24-bit fixed-point DSP chips; even today, coefficients get quantized,
smoothed, and interpolated when you turn a knob. And a polynomial's roots — your filter's poles —
grow *exquisitely* more sensitive to coefficient error as its order rises, especially when roots
crowd together, which is exactly what our low-frequency poles do near $z = 1$.

The experiment below is provided — your job is to run it, poke at it, and **explain what you see
in your `TECHNICAL.md`** (with these plots). It does two things:

1. **Factors the rumble filter into true biquads first**, because even a third-order section is a
   small liability. The Butterworth-3 polynomial factors as $B_3(s) = (s + 1)(s^2 + s + 1)$, so
   the 18 dB/oct high-pass is exactly a first-order high-pass cascaded with a $Q{=}1$ second-order
   one — this factoring is precisely what production EQs do to *every* high-order section.
2. **Rounds every coefficient to `DECIMALS` decimal places**, in both the biquad cascade and the
   combined filter, then compares responses, pole radii, and — the machine's verdict — impulse
   responses.

The punchline at five decimals: the biquad cascade barely flinches, while the combined filter's
poles slide **outside the unit circle** — the difference equation doesn't just sound wrong, it
**explodes**. Then try `DECIMALS = 6`: *still* unstable. That asymmetry is why the biquad cascade
is the universal shipping format for digital EQ.


In [ ]:
DECIMALS = 5  # five digits: plenty for biquads, fatal for the combined form. Try 4 and 6 too.

# Step 1 — the industry rule: nothing above second order. Factor the third-order
# rumble filter using B3(s) = (s + 1)(s^2 + s + 1): a first-order high-pass
# cascaded with a Q=1 second-order high-pass, both at the same 50 Hz cutoff.
rumble_factored = [
    bilinear_transform(np.array([1.0, 0.0]), np.array([1.0, 1.0]), 50.0, SAMPLE_RATE),
    bilinear_transform(np.array([1.0, 0.0, 0.0]), np.array([1.0, 1.0, 1.0]), 50.0, SAMPLE_RATE),
]
true_biquads = rumble_factored + digital_sections[1:]

# Step 2 — round every coefficient, everywhere, and compare.
quantize = lambda arr: np.round(np.asarray(arr, dtype=float), DECIMALS)
biquads_q = [(quantize(b), quantize(a)) for b, a in true_biquads]
b_q, a_q = quantize(b_full), quantize(a_full)

H_cascade_q = np.ones_like(FREQS, dtype=complex)
for b, a in biquads_q:
    H_cascade_q *= digital_response(b, a, FREQS, SAMPLE_RATE)
H_combined_q = digital_response(b_q, a_q, FREQS, SAMPLE_RATE)

passband = FREQS >= 100
dev_casc_q = np.max(np.abs(db(H_cascade_q) - db(H_combined))[passband])
dev_comb_q = np.max(np.abs(db(H_combined_q) - db(H_combined))[passband])
pole_casc_q = max(np.max(np.abs(np.roots(a))) for _, a in biquads_q)
pole_comb_q = np.max(np.abs(np.roots(a_q)))
print(f"rounded to {DECIMALS} decimal places:")
print(f"  biquad cascade — largest pole radius {pole_casc_q:.6f}, "
      f"passband response off by {dev_casc_q:.3f} dB  (shrugs)")
print(f"  combined form  — largest pole radius {pole_comb_q:.6f}, "
      f"passband response off by {dev_comb_q:.1f} dB"
      + ("   💥 pole OUTSIDE the unit circle — unstable!" if pole_comb_q >= 1 else ""))

# Step 3 — the machine's verdict: feed both quantized filters an impulse.
impulse = np.zeros(1500)
impulse[0] = 1.0
ir_casc_q = impulse
for b, a in biquads_q:
    ir_casc_q = apply_filter(b, a, ir_casc_q)
ir_comb_q = apply_filter(b_q, a_q, impulse)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
ax1.semilogx(FREQS, db(H_combined), color="darkgoldenrod", lw=3, alpha=0.5, label="exact")
ax1.semilogx(FREQS, db(H_cascade_q), color="tab:blue", lw=1.4, ls="--",
             label=f"biquad cascade @ {DECIMALS} decimals")
ax1.semilogx(FREQS, db(H_combined_q), color="tab:red", lw=1.4, ls=":",
             label=f"combined @ {DECIMALS} decimals")
ax1.set_title("Response after coefficient rounding")
ax1.set_xlabel("frequency (Hz)")
ax1.set_ylabel("gain (dB)")
ax1.grid(alpha=0.3, which="both")
ax1.legend(fontsize=8)

ax2.semilogy(np.abs(ir_casc_q) + 1e-12, color="tab:blue", lw=1.0,
             label="quantized biquad cascade")
ax2.semilogy(np.abs(ir_comb_q) + 1e-12, color="tab:red", lw=1.0,
             label="quantized combined")
ax2.set_title("Impulse response magnitude (log scale)")
ax2.set_xlabel("sample")
ax2.set_ylabel("|y[n]|")
ax2.grid(alpha=0.3)
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()


## 🔊 Task 7 — Hear it

Plots are proof for the graders. This part is proof for *you*.

The cell below synthesizes a rough mix (drums, bass, a pad) and then dulls it with a gentle
low-pass, the way a bad cassette copy would. Then we run it through the resurrected channel strip:
rumble filtered, lows firmed, presence lifted, air restored — the move engineers have reached for a
1073 to make for fifty years.

**You are encouraged to swap in a dry recording of your own** —
`pq.Audio.from_file("your_mix.wav")` (mix to mono, mind the sample rate).

One detail worth noticing: we process through the **cascade of sections**, not the combined
9th-order equation — for exactly the reason Task 6 just showed you.


In [ ]:
rng = np.random.default_rng(322)
BEAT = 60 / 96  # 96 BPM
BARS = 4
tape = np.zeros(int(BARS * 4 * BEAT * SAMPLE_RATE))


def _env(n, decay):
    return np.exp(-decay * np.arange(n) / SAMPLE_RATE)


def _place(start_beat, sound):
    i = int(start_beat * BEAT * SAMPLE_RATE)
    j = min(len(tape), i + len(sound))
    tape[i:j] += sound[: j - i]


def _kick():
    t = np.arange(int(0.25 * SAMPLE_RATE)) / SAMPLE_RATE
    phase = 45 * t + (110 / 25) * (1 - np.exp(-25 * t))  # a 155 Hz thump falling to 45 Hz
    return np.sin(2 * np.pi * phase) * _env(len(t), 9)


def _snare():
    n = int(0.18 * SAMPLE_RATE)
    return rng.standard_normal(n) * _env(n, 22) * 0.45


def _hat():
    n = int(0.05 * SAMPLE_RATE)
    return np.diff(rng.standard_normal(n), prepend=0.0) * _env(n, 70) * 0.30


def _saw(freq, dur, amp, decay):
    t = np.arange(int(dur * SAMPLE_RATE)) / SAMPLE_RATE
    wave = sum(np.sin(2 * np.pi * freq * k * t) / k for k in range(1, 12))
    return amp * wave * _env(len(t), decay)


bass_roots = [55.00, 65.41, 73.42, 82.41]  # A1, C2, D2, E2 — one per bar
for bar, root in enumerate(bass_roots):
    t0 = bar * 4
    for beat in range(4):
        _place(t0 + beat, _kick() if beat % 2 == 0 else _snare())
        _place(t0 + beat, _hat())
        _place(t0 + beat + 0.5, _hat())
        _place(t0 + beat, _saw(root * (2 if beat == 2 else 1), 0.55 * BEAT, 0.35, 4))
    for ratio in [4.0, 4.0 * 2 ** (3 / 12), 4.0 * 2 ** (7 / 12)]:  # minor triad pad
        _place(t0, _saw(root * ratio * 1.002, 4 * BEAT, 0.035, 0.6))
        _place(t0, _saw(root * ratio / 1.002, 4 * BEAT, 0.035, 0.6))

tape *= 0.9 / np.max(np.abs(tape))

# A bad cassette copy: a gentle one-pole low-pass dulls everything.
alpha = 1 - np.exp(-2 * np.pi * 2200 / SAMPLE_RATE)
tape = apply_filter([alpha], [1.0, alpha - 1.0], tape)

print("The dull rough mix, before:")
pq.play(pq.Audio(tape, SAMPLE_RATE), normalize=True)


In [ ]:
restored = tape
for b, a in digital_sections:
    restored = apply_filter(b, a, restored)

print("The same mix, through your resurrected channel strip:")
pq.play(pq.Audio(restored, SAMPLE_RATE), normalize=True)


## 🚀 You're in the business now

Step back and look at what you built. Nothing in your four functions knows anything about the
1073 — **you built a compiler**, and the 1073-style channel strip was merely its first program.
Feed it any list of analog sections and it will carry them across the bridge just the same.

And here is the part worth saying out loud: **none of this was a dramatization.** Universal Audio,
Brainworx, Neural DSP, Waves — there are entire companies whose products are made exactly this way:
take a revered analog circuit, derive its continuous-time transfer function, and discretize it,
very often with the very bilinear transform you just implemented. The plugins on every professional
mixing session in the world come out of the pipeline sitting in this notebook. You didn't do a
homework-sized imitation of the real thing. You did the real thing, small.

To finish the project, head to the Project 4 page on the course site and check your work against
the Direction 2 requirements: your four functions, the verification and fragility plots,
`TECHNICAL.md`, and the demo video (the target curve, the four curves agreeing,
the combined filter exploding, and a before/after listen make a great story arc).

### If you're hungry for more (optional, not required)

- **Design an EQ of your own** — pick your bands, name it, compile it, and run a recording of
  yours through it.
- Support **more section types** — notch, band-pass, low-pass — or Butterworth sections of
  arbitrary order.
- Generalize Task 6's hand-factoring: **factor any high-order section into biquads
  programmatically** (`np.roots`, pair the complex-conjugate roots), like the real ones do.
- The final boss, with a real published answer to check yourself against: implement the analog
  transfer function from Yeh & Smith's *"Discretization of the '59 Fender Bassman Tone Stack"*
  (DAFx 2006) and compile a real amplifier's tone circuit through your own pipeline.

Rupert Neve spent half a century insisting you could hear the difference. Now you can compute it.
🎚️
